In [12]:
print("Removing saved progress:", CHECKPOINT)
CHECKPOINT.unlink(missing_ok=True)

Removing saved progress: C:\Users\aanand37\qwen_visibility\head_visibility_fixed.jsonl


In [2]:
from types import SimpleNamespace

CFG = SimpleNamespace(
    # --- Data ---
    # Auto-detected below, but you can hard-code if needed:
    input_root   = "Q:\\aanand37\\Manuscript\\Snapshot Wisconsin\\data\\Deer_clipped",   #\\test       # e.g. "/kaggle/input/snapshot-wi-oh-deer"
    csv_path     = "Q:\\aanand37\\Manuscript\\Snapshot Wisconsin\\data\\Snapshot_WI-Oh_Deer_data-v1.csv",          # auto-globbed if None
    photos_dir   = "Q:\\aanand37\\Manuscript\\Snapshot Wisconsin\\data\\Deer_clipped",          # auto-globbed if None

    # --- Model ---
    model_name   = "efficientnet_b0",   # any timm model; try "convnext_nano" for +accuracy
    img_size     = 256,
    bbox_pad     = 0.15,          # expand the bounding box by 15% before cropping
    use_bbox     = True,          # crop to the deer using the provided box

    # --- Training ---
    epochs       = 12,
    batch_size   = 32,
    lr           = 3e-4,
    weight_decay = 1e-4,
    warmup_frac  = 0.1,
    label_smooth = 0.05,
    val_frac     = 0.15,
    n_folds      = 1,             # set >1 to run stratified K-fold CV
    seed         = 42,
    num_workers  = 2,
    amp          = True,          # mixed precision (fast on T4)

    # --- Inference ---
    tune_thresholds = True,       # search decision thresholds to maximize macro-F1
    tta          = False,         # horizontal-flip TTA (small accuracy gain, 2x slower)

    # --- Output ---
    out_dir      = "/kaggle/working",
    model_file   = "oh_deer_model.pt",
    pred_file    = "predictions.csv",
)
print(CFG)

namespace(input_root='Q:\\aanand37\\Manuscript\\Snapshot Wisconsin\\data\\Deer_clipped', csv_path='Q:\\aanand37\\Manuscript\\Snapshot Wisconsin\\data\\Snapshot_WI-Oh_Deer_data-v1.csv', photos_dir='Q:\\aanand37\\Manuscript\\Snapshot Wisconsin\\data\\Deer_clipped', model_name='efficientnet_b0', img_size=256, bbox_pad=0.15, use_bbox=True, epochs=12, batch_size=32, lr=0.0003, weight_decay=0.0001, warmup_frac=0.1, label_smooth=0.05, val_frac=0.15, n_folds=1, seed=42, num_workers=2, amp=True, tune_thresholds=True, tta=False, out_dir='/kaggle/working', model_file='oh_deer_model.pt', pred_file='predictions.csv')


In [3]:
import sys, subprocess
def _pip(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_pip("timm")

import os, glob, random, time, math, warnings
import numpy as np
import pandas as pd
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
warnings.filterwarnings("ignore")
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG.seed)

if not torch.cuda.is_available():
    raise RuntimeError(
        f"GPU training is required, but CUDA is unavailable. Python: {sys.executable}; "
        f"PyTorch: {torch.__version__}; compiled CUDA: {torch.version.cuda}. "
        "Use a CUDA-enabled PyTorch environment with an NVIDIA GPU."
    )
torch.cuda.set_device(0)
DEVICE = "cuda"
print(f"torch {torch.__version__} | timm {timm.__version__} | device: {DEVICE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | timm 1.0.29 | device: cuda
GPU: NVIDIA T1000


In [4]:
from pathlib import Path
import pandas as pd

root = Path(r"Q:\aanand37\Manuscript\Snapshot Wisconsin\data") #\test

CFG.input_root = str(root)
CFG.csv_path = str(root / "Snapshot_WI-Oh_Deer_data-v1.csv")
#CFG.photos_dir = str(root / "Snapshot_WI-Oh_Deer_Photos" / "Deer")
CFG.photos_dir = str(root / "Deer_clipped")
if not Path(CFG.csv_path).is_file():
    raise FileNotFoundError(f"CSV not found: {CFG.csv_path}")

if not Path(CFG.photos_dir).is_dir():
    raise FileNotFoundError(f"Photos folder not found: {CFG.photos_dir}")

print("input_root:", CFG.input_root)
print("csv_path  :", CFG.csv_path)
print("photos_dir:", CFG.photos_dir)

df = pd.read_csv(CFG.csv_path)

print("\nCSV shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

input_root: Q:\aanand37\Manuscript\Snapshot Wisconsin\data
csv_path  : Q:\aanand37\Manuscript\Snapshot Wisconsin\data\Snapshot_WI-Oh_Deer_data-v1.csv
photos_dir: Q:\aanand37\Manuscript\Snapshot Wisconsin\data\Deer_clipped

CSV shape: (4880, 10)
Columns: ['filename', 'year', 'month', 'bbox_origin_x', 'bbox_origin_y', 'bbox_width', 'bbox_height', 'bbox_confidence', 'antler_status', 'age_status']


,filename,year,month,bbox_origin_x,bbox_origin_y,bbox_width,bbox_height,bbox_confidence,antler_status,age_status
0,SSWI000000027884533B.jpg,2022,6,0.2687,0.1066,0.5043,0.6933,0.966,ANTLERLESS,YOUNG
1,SSWI000000019469717A.jpg,2020,9,0.4831,0.2608,0.5168,0.7275,0.978,ANTLERLESS,YOUNG
2,SSWI000000020749648B.jpg,2021,1,0.0000,0.4750,0.6606,0.5216,0.978,ANTLERLESS,YOUNG
3,SSWI000000010695719B.jpg,2018,8,0.2587,0.4591,0.1531,0.3833,0.969,ANTLERLESS,YOUNG
4,SSWI000000020110273C.jpg,2020,8,0.2068,0.7733,0.2693,0.2241,0.956,ANTLERLESS,YOUNG


In [5]:
import torch

print("PyTorch:", torch.__version__)
print("Compiled CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Compiled CUDA: 12.8
CUDA available: True
GPU: NVIDIA T1000


In [6]:
# %pip install -q openai pillow pandas tqdm

import base64
import io
import json
import time
from pathlib import Path

import pandas as pd
from PIL import Image, ImageOps, ImageDraw
from tqdm import tqdm
from openai import OpenAI


# ============================================================
# CONFIGURATION
# ============================================================

client = OpenAI(
    base_url="https://llm-gw01.doit.wisc.edu/v1",
    timeout=300,
    max_retries=2,
)

MODEL = "qwen3.8-27b"

PHOTOS_DIR = Path(CFG.photos_dir)

OUTPUT_DIR = Path.cwd() / "qwen_visibility"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT = OUTPUT_DIR / "head_visibility.jsonl"


# ------------------------------------------------------------
# Image settings
# ------------------------------------------------------------

MAX_IMAGE_SIZE = 640 #1024
JPEG_QUALITY = 65


# ------------------------------------------------------------
# Model settings
# ------------------------------------------------------------

# Qwen may use reasoning/output tokens internally.
MAX_TOKENS = 32

# Retry an individual image if the response cannot be parsed.
MAX_ATTEMPTS = 3

# Delay between retries.
RETRY_DELAY = 5


# ------------------------------------------------------------
# Only TWO valid classes
# ------------------------------------------------------------

VALID_LABELS = {
    "visible",
    "not_visible",
}


# ============================================================
# PROMPT
# ============================================================

PROMPT = """
Inspect this trail-camera photograph.

The red rectangle identifies the target deer.

The rectangle is only an approximate bounding box around the target
animal. Inspect the whole photograph, including areas outside the
rectangle. Do not treat the rectangle boundaries as photograph
boundaries.

Ignore all other deer.

Your task is to decide whether the HEAD of the target deer is visible.

Use exactly one of these two classes:

visible
= enough of the target deer's head can be seen to recognize that
the head is present.

The head may be:
- turned away
- partially occluded
- partly outside the red rectangle
- dark
- somewhat blurred
- small in the image

As long as the head itself can still be visually identified,
classify it as visible.

not_visible
= the target deer's head cannot be visually identified because it is:
- completely outside the photograph
- completely hidden behind vegetation or another object
- completely obscured
- not distinguishable at all

Do not infer head visibility from:
- antlers
- body position
- sex
- age
- body orientation
- other deer

You MUST choose one of the two classes.

Return ONLY one exact label:

visible
not_visible
""".strip()


# ============================================================
# IMAGE PREPARATION
# ============================================================

def prepare_image(row):
    """
    Load the full trail-camera image, resize it, and draw a red
    rectangle around the target deer.

    The full frame is retained so the model can inspect areas
    outside the approximate bounding box.
    """

    filename = Path(
        str(row["filename"]).strip()
    ).name

    path = PHOTOS_DIR / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Image not found: {path}"
        )

    # --------------------------------------------------------
    # Load image and respect EXIF orientation
    # --------------------------------------------------------

    with Image.open(path) as source:
        image = ImageOps.exif_transpose(
            source
        ).convert("RGB")

    # --------------------------------------------------------
    # Resize while preserving aspect ratio
    # --------------------------------------------------------

    image.thumbnail(
        (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE),
        Image.Resampling.LANCZOS,
    )

    width, height = image.size


    # --------------------------------------------------------
    # Normalized bounding box
    # --------------------------------------------------------

    x = float(row["bbox_origin_x"])
    y = float(row["bbox_origin_y"])
    bw = float(row["bbox_width"])
    bh = float(row["bbox_height"])


    # --------------------------------------------------------
    # Validate coordinates
    # --------------------------------------------------------

    if not (
        0 <= x < 1
        and 0 <= y < 1
        and 0 < bw <= 1
        and 0 < bh <= 1
    ):
        raise ValueError(
            "Invalid normalized bounding box: "
            f"x={x}, y={y}, w={bw}, h={bh}"
        )


    # --------------------------------------------------------
    # Convert normalized coordinates to pixels
    # --------------------------------------------------------

    x1 = max(
        0,
        int(x * width)
    )

    y1 = max(
        0,
        int(y * height)
    )

    x2 = min(
        width - 1,
        int((x + bw) * width)
    )

    y2 = min(
        height - 1,
        int((y + bh) * height)
    )


    if x2 <= x1 or y2 <= y1:
        raise ValueError(
            "Invalid bounding box after conversion: "
            f"{(x1, y1, x2, y2)}"
        )


    # --------------------------------------------------------
    # Make rectangle sufficiently visible
    # --------------------------------------------------------

    rectangle_width = max(
        2,
        round(
            max(width, height) / 500
        ),
    )

    draw = ImageDraw.Draw(image)

    draw.rectangle(
        (x1, y1, x2, y2),
        outline="red",
        width=rectangle_width,
    )

    return image


# ============================================================
# IMAGE -> BASE64
# ============================================================

def image_to_base64(image):
    """
    Compress the image to JPEG before sending it to the API.
    """

    buffer = io.BytesIO()

    image.save(
        buffer,
        format="JPEG",
        quality=JPEG_QUALITY,
        optimize=True,
    )

    encoded = base64.b64encode(
        buffer.getvalue()
    ).decode("ascii")

    return encoded


# ============================================================
# RESPONSE PARSING
# ============================================================

def parse_label(text):
    """
    Extract either:

        visible
        not_visible

    from the model response.

    Handles:
        visible
        not_visible
        not visible
        "visible"
        visible.
        {"head_visibility": "visible"}
        longer accidental responses
        partially truncated responses containing the label
    """

    if text is None:
        raise ValueError(
            "Empty model response"
        )


    # --------------------------------------------------------
    # Normalize
    # --------------------------------------------------------

    raw = str(text)

    label = raw.strip().lower()

    # Remove Markdown backticks.
    label = label.replace("`", "")

    # --------------------------------------------------------
    # 1. Exact match
    # --------------------------------------------------------

    if label in VALID_LABELS:
        return label


    # --------------------------------------------------------
    # 2. Remove punctuation
    # --------------------------------------------------------

    cleaned = label.strip(
        " \n\t.,:;!?\"'"
    )

    if cleaned in VALID_LABELS:
        return cleaned


    # --------------------------------------------------------
    # 3. JSON response
    # --------------------------------------------------------

    try:

        obj = json.loads(label)

        if isinstance(obj, dict):

            candidate = obj.get(
                "head_visibility"
            )

            if candidate is not None:

                candidate = (
                    str(candidate)
                    .strip()
                    .lower()
                )

                candidate = (
                    candidate
                    .replace("-", "_")
                    .replace(" ", "_")
                )

                if candidate in VALID_LABELS:
                    return candidate

    except (
        json.JSONDecodeError,
        TypeError,
        ValueError,
    ):
        pass


    # --------------------------------------------------------
    # 4. Normalize separators
    # --------------------------------------------------------

    normalized = (
        label
        .replace("-", "_")
        .replace(" ", "_")
    )


    # --------------------------------------------------------
    # IMPORTANT:
    # Check NOT_VISIBLE before VISIBLE because
    # "not_visible" contains the word "visible".
    # --------------------------------------------------------

    if "not_visible" in normalized:
        return "not_visible"


    # --------------------------------------------------------
    # Common natural-language alternatives
    # --------------------------------------------------------

    not_visible_phrases = [
        "not visible",
        "head is not visible",
        "head not visible",
        "cannot see the head",
        "head cannot be seen",
        "head isn't visible",
        "head is invisible",
        "head cannot be identified",
    ]

    for phrase in not_visible_phrases:
        if phrase in label:
            return "not_visible"


    # --------------------------------------------------------
    # VISIBLE
    # --------------------------------------------------------

    visible_phrases = [
        "head is visible",
        "head visible",
        "visible",
    ]

    for phrase in visible_phrases:
        if phrase in label:
            return "visible"


    # --------------------------------------------------------
    # Nothing useful found
    # --------------------------------------------------------

    raise ValueError(
        "No visibility label found in model response: "
        f"{raw!r}"
    )


# ============================================================
# CLASSIFY ONE IMAGE
# ============================================================

def classify_head(row):
    """
    Classify one image as:

        visible
        not_visible

    The API request is retried if the model output cannot be parsed.

    finish_reason == 'length' does NOT automatically cause failure.

    If all attempts fail, the image receives the conservative fallback
    classification:

        not_visible

    and fallback=True is stored so those cases can be inspected later.
    """

    # --------------------------------------------------------
    # Prepare image once
    # --------------------------------------------------------

    image = prepare_image(row)

    encoded = image_to_base64(
        image
    )


    last_error = None
    last_response = None
    last_finish_reason = None


    # ========================================================
    # Retry loop
    # ========================================================

    for attempt in range(
        1,
        MAX_ATTEMPTS + 1
    ):

        try:

            response = (
                client.chat.completions.create(
                    model=MODEL,

                    temperature=0,

                    max_tokens=MAX_TOKENS,

                    messages=[
                        {
                            "role": "system",

                            "content": (
                                "You are performing binary "
                                "wildlife-image classification. "
                                "You MUST classify every image "
                                "as exactly one of these two "
                                "labels: visible or not_visible. "
                                "Return only the label and "
                                "nothing else."
                            ),
                        },

                        {
                            "role": "user",

                            "content": [
                                {
                                    "type": "text",
                                    "text": PROMPT,
                                },

                                {
                                    "type": "image_url",

                                    "image_url": {
                                        "url": (
                                            "data:image/jpeg;base64,"
                                            + encoded
                                        )
                                    },
                                },
                            ],
                        },
                    ],
                )
            )


            # ------------------------------------------------
            # Get first returned choice
            # ------------------------------------------------

            choice = response.choices[0]

            last_finish_reason = (
                choice.finish_reason
            )

            text = (
                choice.message.content
            )

            last_response = text


            # =================================================
            # IMPORTANT:
            #
            # We DO NOT do:
            #
            # if choice.finish_reason == "length":
            #     raise ...
            #
            # Instead, parse whatever text was returned.
            # =================================================

            label = parse_label(
                text
            )


            # ------------------------------------------------
            # Successful classification
            # ------------------------------------------------

            return {
                "head_visibility": label,
                "fallback": False,
                "attempts": attempt,
                "finish_reason": last_finish_reason,
                "error": None,
            }


        except Exception as exc:

            last_error = exc


            # ------------------------------------------------
            # Retry if attempts remain
            # ------------------------------------------------

            if attempt < MAX_ATTEMPTS:

                time.sleep(
                    RETRY_DELAY
                )


    # ========================================================
    # ALL ATTEMPTS FAILED
    #
    # User requested every image to receive a binary class.
    #
    # Conservative fallback:
    # not_visible
    # ========================================================

    return {
        "head_visibility": "not_visible",

        "fallback": True,

        "attempts": MAX_ATTEMPTS,

        "finish_reason": last_finish_reason,

        "error": (
            f"{type(last_error).__name__}: "
            f"{str(last_error)[:500]}"
        ),

        "raw_response": (
            str(last_response)[:1000]
            if last_response is not None
            else None
        ),
    }



### All in one

In [13]:
# ============================================================
# QWEN HEAD-VISIBILITY SCREENING — DROP-IN REPLACEMENT
# ============================================================
#
# Replaces the previous Qwen/API screening cells.
#
# IMPORTANT FIXES:
#   1. Only 1 API request at a time by default.
#   2. OpenAI SDK retries disabled — we control retries ourselves.
#   3. Smaller 640 px JPEG images.
#   4. Very small output-token budget.
#   5. Attempts to disable Qwen thinking/reasoning.
#   6. Exponential backoff after timeout/rate-limit errors.
#   7. API failures are NOT silently called "not_visible".
#   8. Failed rows are NOT treated as successful cached results.
#   9. Text + vision preflight before processing 4,880 images.
#  10. Resumable JSONL checkpoint.
#
# If needed:
# %pip install -q openai pillow pandas tqdm httpx
# ============================================================


# ============================================================
# IMPORTS
# ============================================================

import base64
import io
import json
import random
import re
import time
from concurrent.futures import (
    FIRST_COMPLETED,
    ThreadPoolExecutor,
    wait,
)
from datetime import datetime
from pathlib import Path

import httpx
import pandas as pd
from PIL import Image, ImageDraw, ImageOps
from tqdm import tqdm

from openai import OpenAI


# ============================================================
# CHECK INPUTS
# ============================================================

if "CFG" not in globals():
    raise RuntimeError(
        "CFG does not exist. Run your earlier configuration/data-loading "
        "cells first."
    )

if "df" not in globals():
    print("df not found — loading from CFG.csv_path")
    df = pd.read_csv(CFG.csv_path)

df = df.reset_index(drop=True)

required_columns = [
    "filename",
    "bbox_origin_x",
    "bbox_origin_y",
    "bbox_width",
    "bbox_height",
]

missing_columns = [
    c for c in required_columns
    if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )


# ============================================================
# CONFIGURATION
# ============================================================

MODEL = "qwen3.8-27b"

PHOTOS_DIR = Path(CFG.photos_dir)

if not PHOTOS_DIR.is_dir():
    raise FileNotFoundError(
        f"Photo folder does not exist:\n{PHOTOS_DIR}"
    )


# ------------------------------------------------------------
# Output folder
# ------------------------------------------------------------

OUTPUT_DIR = Path.cwd() / "qwen_visibility"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# IMPORTANT:
# This intentionally uses a NEW checkpoint filename.
#
# It will therefore NOT reuse the old checkpoint in which all
# 4,880 timeout failures were saved as not_visible.
#
CHECKPOINT = (
    OUTPUT_DIR
    / "head_visibility_fixed.jsonl"
)


# ------------------------------------------------------------
# Image size
# ------------------------------------------------------------
#
# Your old code used 1024.
#
# 640 substantially reduces the multimodal payload while still
# providing enough detail for this binary screening task.
#
# If the gateway works reliably and you later want more detail,
# try 768.
# ------------------------------------------------------------

MAX_IMAGE_SIZE = 1024#640

JPEG_QUALITY = 100#65


# ------------------------------------------------------------
# API / generation settings
# ------------------------------------------------------------

# Binary answer only needs a tiny output.
#
# 32 allows a little tolerance if the server emits formatting,
# while remaining far below the previous 1024.
MAX_TOKENS = 32

# We control retries ourselves.
MAX_ATTEMPTS = 3

# Base delay:
# attempt 1 failure -> ~5 sec
# attempt 2 failure -> ~10 sec
# attempt 3 failure -> stop
RETRY_BASE_DELAY = 5

# Long enough for a vision call, but not so long that one dead
# request occupies a worker indefinitely.
REQUEST_TIMEOUT_SECONDS = 240

# ------------------------------------------------------------
# CONCURRENCY
# ------------------------------------------------------------
#
# START WITH 1.
#
# DO NOT derive this from os.cpu_count().
# This workload is limited by the remote API, not your CPU.
#
MAX_WORKERS = 1


# ------------------------------------------------------------
# Preflight
# ------------------------------------------------------------

RUN_PREFLIGHT = True

# Full dataset.
#
# For an initial short test instead, use:
#
# N_TO_PROCESS = 20
#
N_TO_PROCESS = None


# ------------------------------------------------------------
# Try disabling Qwen reasoning/thinking.
#
# vLLM/Qwen deployments commonly support:
#
# chat_template_kwargs={"enable_thinking": False}
#
# If your UW gateway rejects this parameter, the code detects
# that and automatically falls back to a normal request.
# ------------------------------------------------------------

TRY_DISABLE_THINKING = True


VALID_LABELS = {
    "visible",
    "not_visible",
}


# ============================================================
# OPENAI CLIENT
# ============================================================

timeout_config = httpx.Timeout(
    timeout=REQUEST_TIMEOUT_SECONDS,
    connect=20.0,
    read=REQUEST_TIMEOUT_SECONDS,
    write=60.0,
    pool=20.0,
)

client = OpenAI(
    base_url="https://llm-gw01.doit.wisc.edu/v1",

    # Uses the same authentication/environment you were already
    # using successfully for the UW gateway.
    timeout=timeout_config,

    # CRITICAL:
    # Do not let the SDK perform another set of hidden retries.
    max_retries=0,
)


# ============================================================
# PROMPT
# ============================================================

PROMPT = """
Inspect this trail-camera photograph and classify HEAD VISIBILITY
for the TARGET deer.

TARGET IDENTIFICATION:
- If a red rectangle is present, it identifies the target deer.
- The rectangle is approximate: inspect the entire photograph,
  including outside the rectangle, to locate that deer's head.
- If there is only one deer and no rectangle, classify that deer.
- Ignore other deer.

The task is to detect ANY recognizable part of the head,
NOT to determine whether the face or entire head is clearly visible.

Return "visible" if ANY recognizable portion of the target deer's
head is visible, including the top, back, side, forehead, cheek,
muzzle, or an identifiable ear attached to the head.
- The eyes, nose, mouth, and full face do NOT need to be visible.
- A head turned away from the camera still counts as visible.
- A lowered head while grazing still counts as visible.
- A head partly hidden by grass, ferns, branches, or the body
  still counts as visible if any head portion can be identified.
- A dark, blurred, small, or partly cropped head still counts
  as visible when a head portion remains recognizable.
- Do not require a clear facial outline or unobstructed view.

Examples that MUST be classified as "visible":
- A grazing deer with its ears and upper head visible above grass,
  even when its muzzle and eyes are hidden.
- A deer looking away with the back or side of its head and ears
  visible, even when ferns obscure its face.

Return "not_visible" ONLY when NO recognizable portion of the
target deer's head or attached ears is visible:
- The entire head is outside the photograph.
- The entire head is hidden behind vegetation, the body,
  another animal, or another obstruction.
- The head region is completely indistinguishable.

Antlers alone do not establish head visibility.
Use the body and neck to locate the target's head, but do not
assume the head is visible solely from body posture or orientation.

Before deciding, check whether a partially visible head has been
mistaken for a missing head. Partial visibility is "visible".

Return ONLY one exact label:
visible
not_visible
""".strip()


# ============================================================
# IMAGE PREPARATION
# ============================================================

def prepare_image(row):
    """
    Load the complete trail-camera frame, resize it while preserving
    aspect ratio, and draw the target-deer bounding box in red.
    """

    filename = Path(
        str(row["filename"]).strip()
    ).name

    path = PHOTOS_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(
            f"Image not found: {path}"
        )

    # --------------------------------------------------------
    # Load / EXIF orientation
    # --------------------------------------------------------

    with Image.open(path) as source:
        image = ImageOps.exif_transpose(
            source
        ).convert("RGB")

    # --------------------------------------------------------
    # Downsample BEFORE drawing the rectangle.
    # --------------------------------------------------------

    image.thumbnail(
        (
            MAX_IMAGE_SIZE,
            MAX_IMAGE_SIZE,
        ),
        Image.Resampling.LANCZOS,
    )

    width, height = image.size


    # --------------------------------------------------------
    # Normalized bounding box
    # --------------------------------------------------------

    x = float(row["bbox_origin_x"])
    y = float(row["bbox_origin_y"])
    bw = float(row["bbox_width"])
    bh = float(row["bbox_height"])


    values = [x, y, bw, bh]

    if any(pd.isna(v) for v in values):
        raise ValueError(
            f"Bounding box contains NA: {values}"
        )

    if not (
        0 <= x < 1
        and 0 <= y < 1
        and 0 < bw <= 1
        and 0 < bh <= 1
    ):
        raise ValueError(
            "Invalid normalized bounding box: "
            f"x={x}, y={y}, "
            f"w={bw}, h={bh}"
        )


    # --------------------------------------------------------
    # Convert coordinates
    # --------------------------------------------------------

    x1 = max(
        0,
        int(round(x * width)),
    )

    y1 = max(
        0,
        int(round(y * height)),
    )

    x2 = min(
        width - 1,
        int(round((x + bw) * width)),
    )

    y2 = min(
        height - 1,
        int(round((y + bh) * height)),
    )

    if x2 <= x1 or y2 <= y1:
        raise ValueError(
            "Invalid bounding box after conversion: "
            f"{(x1, y1, x2, y2)}"
        )


    # --------------------------------------------------------
    # Draw rectangle
    # --------------------------------------------------------

    rectangle_width = max(
        3,
        round(
            max(width, height) / 220
        ),
    )

    draw = ImageDraw.Draw(image)

    draw.rectangle(
        (
            x1,
            y1,
            x2,
            y2,
        ),
        outline="red",
        width=rectangle_width,
    )

    return image


# ============================================================
# JPEG -> BASE64
# ============================================================

def image_to_base64(image):
    """
    Convert PIL image into a compact JPEG base64 payload.

    Returns
    -------
    encoded : str
        Base64 encoded JPEG.

    byte_size : int
        Actual JPEG size before base64 encoding.
    """

    buffer = io.BytesIO()

    image.save(
        buffer,
        format="JPEG",
        quality=JPEG_QUALITY,
        optimize=True,
        subsampling=2,
    )

    raw_bytes = buffer.getvalue()

    encoded = base64.b64encode(
        raw_bytes
    ).decode("ascii")

    return encoded, len(raw_bytes)


# ============================================================
# QWEN REQUEST
# ============================================================

# None = haven't tested support yet.
# True = supported.
# False = gateway rejected it.
_THINKING_SUPPORTED = None
_THINKING_WARNING_PRINTED = False


def _is_bad_request(exc):
    """
    Identify an OpenAI-compatible HTTP 400 response without depending
    on a particular OpenAI package version.
    """

    name = type(exc).__name__.lower()

    status = getattr(
        exc,
        "status_code",
        None,
    )

    return (
        status == 400
        or "badrequest" in name
        or "bad_request" in name
    )


def _create_completion(
    messages,
    max_tokens=MAX_TOKENS,
):
    """
    Send a chat completion request.

    First attempts to disable Qwen thinking. If the gateway rejects
    chat_template_kwargs, all subsequent calls omit it.
    """

    global _THINKING_SUPPORTED
    global _THINKING_WARNING_PRINTED

    common = dict(
        model=MODEL,
        messages=messages,
        temperature=0,
        max_tokens=max_tokens,
    )


    # --------------------------------------------------------
    # First choice:
    # Qwen with thinking disabled
    # --------------------------------------------------------

    if (
        TRY_DISABLE_THINKING
        and _THINKING_SUPPORTED is not False
    ):

        try:

            response = (
                client.chat.completions.create(
                    **common,
                    extra_body={
                        "chat_template_kwargs": {
                            "enable_thinking": False,
                        }
                    },
                )
            )

            _THINKING_SUPPORTED = True

            return response

        except Exception as exc:

            # If this is a normal timeout / server problem,
            # DO NOT issue another immediate duplicate request.
            #
            # Only retry without extra_body when the server
            # explicitly rejected the request as bad input.
            if not _is_bad_request(exc):
                raise

            _THINKING_SUPPORTED = False

            if not _THINKING_WARNING_PRINTED:

                print(
                    "\n[info] Gateway does not appear to support "
                    "enable_thinking=False."
                )

                print(
                    "[info] Continuing with normal Qwen requests.\n"
                )

                _THINKING_WARNING_PRINTED = True


    # --------------------------------------------------------
    # Normal OpenAI-compatible call
    # --------------------------------------------------------

    return client.chat.completions.create(
        **common
    )


# ============================================================
# RESPONSE PARSER
# ============================================================

def parse_visibility_label(raw):
    """
    Convert the model response into one of exactly:

        visible
        not_visible

    Raises ValueError if no reliable answer can be extracted.
    """

    if raw is None:
        raise ValueError(
            "Model returned empty content."
        )

    text = str(raw).strip()

    if not text:
        raise ValueError(
            "Model returned blank content."
        )


    # --------------------------------------------------------
    # Remove a Qwen reasoning block if one appears anyway
    # --------------------------------------------------------

    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.IGNORECASE | re.DOTALL,
    ).strip()


    # --------------------------------------------------------
    # Remove simple markdown/code formatting
    # --------------------------------------------------------

    text = text.replace(
        "```",
        "",
    )

    text = text.replace(
        "`",
        "",
    )

    text = text.strip().lower()


    # --------------------------------------------------------
    # Exact response
    # --------------------------------------------------------

    normalized = re.sub(
        r"[\s\-]+",
        "_",
        text,
    )

    normalized = normalized.strip(
        " .,:;!?\"'"
    )

    if normalized in VALID_LABELS:
        return normalized


    # --------------------------------------------------------
    # Sometimes a model returns:
    #
    # Answer: visible
    #
    # Try each line from the END.
    # --------------------------------------------------------

    lines = [
        line.strip().lower()
        for line in text.splitlines()
        if line.strip()
    ]

    for line in reversed(lines):

        line = re.sub(
            r"^(answer|classification|label)\s*:\s*",
            "",
            line,
        )

        line = re.sub(
            r"[\s\-]+",
            "_",
            line,
        )

        line = line.strip(
            " .,:;!?\"'"
        )

        if line in VALID_LABELS:
            return line


    raise ValueError(
        "Could not reliably parse model response: "
        f"{raw!r}"
    )


# ============================================================
# RETRY LOGIC
# ============================================================

def should_retry(exc):
    """
    Return True for temporary errors such as:
      - timeouts
      - connection failures
      - rate limiting
      - 5xx server errors
      - malformed model response

    Return False for errors that usually will not improve by retrying,
    such as authentication failures or HTTP 400.
    """

    # Our parser uses ValueError. It is reasonable to retry a
    # malformed model output.
    if isinstance(exc, ValueError):
        return True

    name = type(exc).__name__.lower()

    status = getattr(
        exc,
        "status_code",
        None,
    )


    # --------------------------------------------------------
    # Retryable HTTP statuses
    # --------------------------------------------------------

    if status in {
        408,
        409,
        429,
        500,
        502,
        503,
        504,
    }:
        return True


    # --------------------------------------------------------
    # Permanent/fatal statuses
    # --------------------------------------------------------

    if status in {
        400,
        401,
        403,
        404,
        422,
    }:
        return False


    # --------------------------------------------------------
    # Exception class names
    # --------------------------------------------------------

    retry_words = [
        "timeout",
        "connection",
        "ratelimit",
        "rate_limit",
        "internalserver",
        "servererror",
        "apierror",
    ]

    if any(
        word in name
        for word in retry_words
    ):
        return True


    fatal_words = [
        "authentication",
        "permission",
        "notfound",
        "badrequest",
    ]

    if any(
        word in name
        for word in fatal_words
    ):
        return False


    # Unknown error:
    # retry, but only up to MAX_ATTEMPTS.
    return True


# ============================================================
# CLASSIFY ONE IMAGE
# ============================================================

def classify_head(
    row,
    *,
    max_attempts=MAX_ATTEMPTS,
):
    """
    Classify one photograph.

    IMPORTANT:
    If every API attempt fails, head_visibility is None.

    A failed API call is NOT biological evidence that the head
    was not visible.
    """

    # --------------------------------------------------------
    # Prepare image
    # --------------------------------------------------------

    try:

        image = prepare_image(row)

        encoded, jpeg_bytes = (
            image_to_base64(image)
        )

    except Exception as exc:

        return {
            "head_visibility": None,
            "status": "failed",
            "fallback": True,
            "attempts": 0,
            "finish_reason": None,
            "error": (
                f"{type(exc).__name__}: "
                f"{str(exc)[:500]}"
            ),
            "raw_response": None,
            "jpeg_kb": None,
        }


    image_url = (
        "data:image/jpeg;base64,"
        + encoded
    )


    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": PROMPT,
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url,
                    },
                },
            ],
        }
    ]


    last_error = None
    last_response = None
    last_finish_reason = None


    # --------------------------------------------------------
    # Attempts
    # --------------------------------------------------------

    for attempt in range(
        1,
        max_attempts + 1,
    ):

        try:

            response = _create_completion(
                messages,
                max_tokens=MAX_TOKENS,
            )


            if not response.choices:
                raise ValueError(
                    "API response contained no choices."
                )


            choice = response.choices[0]

            last_finish_reason = getattr(
                choice,
                "finish_reason",
                None,
            )

            raw = getattr(
                choice.message,
                "content",
                None,
            )

            last_response = raw


            label = parse_visibility_label(
                raw
            )


            # ================================================
            # SUCCESS
            # ================================================

            return {
                "head_visibility": label,
                "status": "ok",
                "fallback": False,
                "attempts": attempt,
                "finish_reason": (
                    last_finish_reason
                ),
                "error": None,
                "raw_response": (
                    str(raw)[:500]
                ),
                "jpeg_kb": round(
                    jpeg_bytes / 1024,
                    1,
                ),
            }


        except Exception as exc:

            last_error = exc

            # Fatal error -> don't waste two more requests.
            if not should_retry(exc):
                break


            # No sleep after final attempt.
            if attempt < max_attempts:

                delay = min(
                    60,
                    RETRY_BASE_DELAY
                    * (2 ** (attempt - 1)),
                )

                # Small jitter helps if multiple workers are
                # eventually used.
                delay += random.uniform(
                    0,
                    1.5,
                )

                time.sleep(delay)


    # ========================================================
    # FAILURE
    #
    # DO NOT classify this image as not_visible.
    # ========================================================

    error_text = (
        f"{type(last_error).__name__}: "
        f"{str(last_error)[:500]}"
        if last_error is not None
        else "Unknown failure"
    )

    return {
        "head_visibility": None,
        "status": "failed",
        "fallback": True,
        "attempts": max_attempts,
        "finish_reason": last_finish_reason,
        "error": error_text,
        "raw_response": (
            str(last_response)[:500]
            if last_response is not None
            else None
        ),
        "jpeg_kb": round(
            jpeg_bytes / 1024,
            1,
        ),
    }


# ============================================================
# PREFLIGHT 1 — TEXT
# ============================================================

def test_text_endpoint():
    """
    Verify that the model itself responds before attempting vision.
    """

    print(
        "\n"
        + "=" * 70
    )

    print(
        "PREFLIGHT 1/2: TEXT-ONLY REQUEST"
    )

    print(
        "=" * 70
    )

    start = time.perf_counter()

    response = _create_completion(
        [
            {
                "role": "user",
                "content": (
                    "Return exactly the word hello"
                ),
            }
        ],
        max_tokens=16,
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    raw = (
        response
        .choices[0]
        .message
        .content
    )

    print(
        "Response:",
        repr(raw),
    )

    print(
        f"Elapsed: {elapsed:.1f} sec"
    )

    print(
        "TEXT TEST: SUCCESS"
    )

    return raw


# ============================================================
# PREFLIGHT 2 — ONE IMAGE
# ============================================================

def test_vision_endpoint(row):
    """
    Verify that one multimodal request works before running thousands.
    """

    print(
        "\n"
        + "=" * 70
    )

    print(
        "PREFLIGHT 2/2: ONE VISION REQUEST"
    )

    print(
        "=" * 70
    )


    # Show payload information before sending.
    test_image = prepare_image(row)

    _, byte_size = (
        image_to_base64(test_image)
    )

    print(
        "Image dimensions:",
        test_image.size,
    )

    print(
        "JPEG payload:",
        f"{byte_size / 1024:.1f} KB",
    )

    print(
        "Filename:",
        row["filename"],
    )


    start = time.perf_counter()

    # Only ONE attempt for preflight.
    #
    # If one image cannot work, do not launch 4,880 images.
    result = classify_head(
        row,
        max_attempts=1,
    )

    elapsed = (
        time.perf_counter()
        - start
    )


    print(
        "\nResult:"
    )

    print(
        json.dumps(
            result,
            indent=2,
            default=str,
        )
    )

    print(
        f"\nElapsed: {elapsed:.1f} sec"
    )


    if result["status"] != "ok":

        raise RuntimeError(
            "\nVISION PREFLIGHT FAILED.\n\n"
            "The full dataset was NOT started.\n"
            "This means the problem is still with the "
            "multimodal API request/gateway rather than "
            "the batch loop.\n\n"
            f"Error: {result['error']}"
        )


    print(
        "\nVISION TEST: SUCCESS"
    )

    return result


# ============================================================
# CHECKPOINT FUNCTIONS
# ============================================================

def load_checkpoint():
    """
    Read the JSONL checkpoint.

    If the same row appears multiple times, the last record wins.
    """

    saved = {}

    if not CHECKPOINT.exists():
        return saved


    with CHECKPOINT.open(
        "r",
        encoding="utf-8",
    ) as handle:

        for line in handle:

            line = line.strip()

            if not line:
                continue

            try:

                record = json.loads(
                    line
                )

                key = (
                    int(record["row_id"]),
                    str(record["filename"]),
                )

                saved[key] = record

            except Exception:
                # Ignore an interrupted/incomplete JSON line.
                continue


    return saved


def reusable(record):
    """
    ONLY genuine successful model classifications may be reused.

    Failed/fallback records are always retried on a future run.
    """

    return (
        record is not None

        and record.get(
            "status"
        ) == "ok"

        and record.get(
            "head_visibility"
        ) in VALID_LABELS

        and record.get(
            "fallback",
            True,
        ) is False

        and not record.get(
            "error"
        )
    )


# ============================================================
# CLASSIFICATION WRAPPER
# ============================================================

def _classify_record(
    row_id,
    filename,
    row,
):
    """
    Add row metadata to classify_head().
    """

    try:

        result = classify_head(row)

    except Exception as exc:

        result = {
            "head_visibility": None,
            "status": "failed",
            "fallback": True,
            "attempts": 0,
            "finish_reason": None,
            "error": (
                f"{type(exc).__name__}: "
                f"{str(exc)[:500]}"
            ),
            "raw_response": None,
            "jpeg_kb": None,
        }


    record = {
        "row_id": int(row_id),
        "filename": str(filename),
    }

    record.update(result)

    return record


# ============================================================
# SCREEN DATASET
# ============================================================

def screen_images(
    frame,
    *,
    max_workers=MAX_WORKERS,
    flush_every=1,
):
    """
    Screen all requested images.

    Successful classifications are resumed from checkpoint.

    Failed classifications are NOT considered reusable, so simply
    running this function again will retry the failed images.
    """

    if max_workers < 1:
        raise ValueError(
            "max_workers must be >= 1"
        )


    # --------------------------------------------------------
    # Existing checkpoint
    # --------------------------------------------------------

    saved = load_checkpoint()

    print(
        f"\nCheckpoint: {CHECKPOINT}"
    )

    print(
        f"Records already in checkpoint: "
        f"{len(saved):,}"
    )


    records = [
        None
    ] * len(frame)


    counts = {
        "ok": 0,
        "failed": 0,
        "cached": 0,
        "new": 0,
    }


    # --------------------------------------------------------
    # Ensure append starts on fresh line
    # --------------------------------------------------------

    needs_newline = False

    if (
        CHECKPOINT.exists()
        and CHECKPOINT.stat().st_size > 0
    ):

        with CHECKPOINT.open(
            "rb"
        ) as handle:

            handle.seek(
                -1,
                2,
            )

            needs_newline = (
                handle.read(1)
                != b"\n"
            )


    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    with CHECKPOINT.open(
        "a",
        encoding="utf-8",
    ) as handle, tqdm(
        total=len(frame),
        desc="Checking heads",
        unit="img",
        dynamic_ncols=True,
        mininterval=0.5,
    ) as pbar:


        if needs_newline:
            handle.write("\n")


        unflushed = 0


        def accept(
            position,
            record,
            *,
            cached,
        ):

            records[position] = record

            if cached:
                counts["cached"] += 1
            else:
                counts["new"] += 1

            if record.get("status") == "ok":
                counts["ok"] += 1
            else:
                counts["failed"] += 1

            pbar.set_postfix(
                {
                    "ok": counts["ok"],
                    "fail": counts["failed"],
                    "cache": counts["cached"],
                },
                refresh=False,
            )

            pbar.update(1)


        def save_record(
            position,
            key,
            record,
        ):

            nonlocal unflushed

            handle.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

            unflushed += 1

            if unflushed >= flush_every:

                handle.flush()

                unflushed = 0


            saved[key] = record

            accept(
                position,
                record,
                cached=False,
            )


        # ====================================================
        # SEQUENTIAL
        # ====================================================

        if max_workers == 1:

            for position, (
                row_id,
                row,
            ) in enumerate(
                frame.iterrows()
            ):

                filename = str(
                    row["filename"]
                )

                key = (
                    int(row_id),
                    filename,
                )

                old = saved.get(
                    key
                )


                # --------------------------------------------
                # Valid cached model answer
                # --------------------------------------------

                if reusable(old):

                    accept(
                        position,
                        old,
                        cached=True,
                    )

                    continue


                # --------------------------------------------
                # New / previous failure
                # --------------------------------------------

                record = _classify_record(
                    row_id,
                    filename,
                    row,
                )

                save_record(
                    position,
                    key,
                    record,
                )


        # ====================================================
        # OPTIONAL BOUNDED CONCURRENCY
        # ====================================================
        #
        # Do not use this until MAX_WORKERS=1 is proven stable.
        # ====================================================

        else:

            pending = {}

            active_keys = set()


            def collect_one():

                done, _ = wait(
                    list(
                        pending.keys()
                    ),
                    return_when=FIRST_COMPLETED,
                )

                for future in done:

                    position, key = (
                        pending.pop(
                            future
                        )
                    )

                    active_keys.discard(
                        key
                    )

                    try:

                        record = (
                            future.result()
                        )

                    except Exception as exc:

                        row_id, filename = key

                        record = {
                            "row_id": row_id,
                            "filename": filename,
                            "head_visibility": None,
                            "status": "failed",
                            "fallback": True,
                            "attempts": 0,
                            "finish_reason": None,
                            "error": (
                                f"{type(exc).__name__}: "
                                f"{str(exc)[:500]}"
                            ),
                            "raw_response": None,
                            "jpeg_kb": None,
                        }


                    save_record(
                        position,
                        key,
                        record,
                    )


            with ThreadPoolExecutor(
                max_workers=max_workers
            ) as pool:


                for position, (
                    row_id,
                    row,
                ) in enumerate(
                    frame.iterrows()
                ):

                    filename = str(
                        row["filename"]
                    )

                    key = (
                        int(row_id),
                        filename,
                    )


                    # Duplicate key already being processed.
                    while key in active_keys:
                        collect_one()


                    old = saved.get(
                        key
                    )


                    if reusable(old):

                        accept(
                            position,
                            old,
                            cached=True,
                        )

                        continue


                    # Keep the queue bounded.
                    while (
                        len(pending)
                        >= 2 * max_workers
                    ):
                        collect_one()


                    future = pool.submit(
                        _classify_record,
                        row_id,
                        filename,
                        row.copy(),
                    )

                    pending[future] = (
                        position,
                        key,
                    )

                    active_keys.add(
                        key
                    )


                while pending:
                    collect_one()


        handle.flush()


    # ========================================================
    # RESULT DATAFRAME
    # ========================================================

    results = pd.DataFrame(
        records
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        "SCREENING SUMMARY"
    )

    print(
        "=" * 70
    )

    print(
        f"Images requested:       "
        f"{len(results):,}"
    )

    print(
        f"Processed this run:     "
        f"{counts['new']:,}"
    )

    print(
        f"Loaded from checkpoint: "
        f"{counts['cached']:,}"
    )

    print(
        f"Successful model calls: "
        f"{counts['ok']:,}"
    )

    print(
        f"Failed model calls:     "
        f"{counts['failed']:,}"
    )


    if not results.empty:

        print(
            "\nSuccessful classification counts:"
        )

        good = results[
            results["status"] == "ok"
        ]

        print(
            good[
                "head_visibility"
            ].value_counts(
                dropna=False
            )
        )


        failures = results[
            results["status"] != "ok"
        ]

        if len(failures):

            print(
                "\nMost common errors:"
            )

            print(
                failures[
                    "error"
                ].value_counts().head(10)
            )


    return results


# ============================================================
# RUN PREFLIGHT
# ============================================================

print(
    f"Model:          {MODEL}"
)

print(
    f"Images:         {len(df):,}"
)

print(
    f"Photo folder:   {PHOTOS_DIR}"
)

print(
    f"Image max size: {MAX_IMAGE_SIZE}px"
)

print(
    f"JPEG quality:   {JPEG_QUALITY}"
)

print(
    f"Workers:        {MAX_WORKERS}"
)

print(
    f"Checkpoint:     {CHECKPOINT}"
)


if RUN_PREFLIGHT:

    # --------------------------------------------------------
    # Text endpoint
    # --------------------------------------------------------

    try:

        test_text_endpoint()

    except Exception as exc:

        raise RuntimeError(
            "\nTEXT PREFLIGHT FAILED.\n\n"
            "The full dataset was NOT started.\n\n"
            "This indicates the UW gateway/model itself is "
            "currently unavailable or the request/authentication "
            "configuration is failing.\n\n"
            f"{type(exc).__name__}: {exc}"
        ) from exc


    # --------------------------------------------------------
    # Vision endpoint
    # --------------------------------------------------------

    test_vision_endpoint(
        df.iloc[0]
    )


    print(
        "\n"
        + "=" * 70
    )

    print(
        "ALL PREFLIGHT TESTS PASSED"
    )

    print(
        "=" * 70
    )

    print(
        "Starting dataset screening...\n"
    )


# ============================================================
# SELECT ROWS
# ============================================================

if N_TO_PROCESS is None:

    work_df = df.copy()

else:

    work_df = df.iloc[
        :N_TO_PROCESS
    ].copy()


# Keep original df indices because row_id is used as the
# checkpoint identity.
print(
    f"Rows to process: {len(work_df):,}"
)


# ============================================================
# RUN SCREENING
# ============================================================

start_all = time.perf_counter()

trial_results = screen_images(
    work_df,
    max_workers=MAX_WORKERS,
    flush_every=1,
)

elapsed_all = (
    time.perf_counter()
    - start_all
)


print(
    "\nTotal screening time: "
    f"{elapsed_all / 60:.1f} minutes"
)


# ============================================================
# EXPORT RESULT TABLE
# ============================================================

timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

results_file = (
    OUTPUT_DIR
    / f"screening_results_fixed_{timestamp}.csv"
)

trial_results.to_csv(
    results_file,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# ALSO ADD RESULTS TO ORIGINAL DATAFRAME
# ============================================================

# Merge by row_id rather than assuming completion order.
merge_cols = [
    "row_id",
    "head_visibility",
    "status",
    "fallback",
    "attempts",
    "finish_reason",
    "error",
]

result_for_merge = (
    trial_results[
        merge_cols
    ]
    .copy()
)


original_with_results = (
    work_df
    .copy()
    .reset_index()
    .rename(
        columns={
            "index": "row_id"
        }
    )
)


original_with_results = (
    original_with_results.merge(
        result_for_merge,
        on="row_id",
        how="left",
    )
)


merged_file = (
    OUTPUT_DIR
    / f"dataset_with_head_visibility_{timestamp}.csv"
)


original_with_results.to_csv(
    merged_file,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# EXPORT FAILED ROWS SEPARATELY
# ============================================================

failed_results = trial_results[
    trial_results["status"] != "ok"
].copy()


failed_file = (
    OUTPUT_DIR
    / f"failed_requests_{timestamp}.csv"
)


failed_results.to_csv(
    failed_file,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 70
)

print(
    "FILES SAVED"
)

print(
    "=" * 70
)

print(
    f"\nResults:\n{results_file}"
)

print(
    f"\nDataset + predictions:\n{merged_file}"
)

print(
    f"\nFailed requests:\n{failed_file}"
)

print(
    f"\nCheckpoint:\n{CHECKPOINT}"
)


print(
    "\n"
    + "=" * 70
)

print(
    "FINAL COUNTS"
)

print(
    "=" * 70
)


print(
    "\nStatus:"
)

print(
    trial_results[
        "status"
    ].value_counts(
        dropna=False
    )
)


print(
    "\nHead visibility:"
)

print(
    trial_results[
        "head_visibility"
    ].value_counts(
        dropna=False
    )
)


# ============================================================
# DISPLAY FIRST RESULTS
# ============================================================

display(
    trial_results.head(20)
)

Model:          qwen3.8-27b
Images:         4,880
Photo folder:   Q:\aanand37\Manuscript\Snapshot Wisconsin\data\Deer_clipped
Image max size: 1024px
JPEG quality:   100
Workers:        1
Checkpoint:     C:\Users\aanand37\qwen_visibility\head_visibility_fixed.jsonl

PREFLIGHT 1/2: TEXT-ONLY REQUEST
Response: 'hello'
Elapsed: 0.1 sec
TEXT TEST: SUCCESS

PREFLIGHT 2/2: ONE VISION REQUEST
Image dimensions: (994, 1024)
JPEG payload: 639.4 KB
Filename: SSWI000000027884533B.jpg

Result:
{
  "head_visibility": "visible",
  "status": "ok",
  "fallback": false,
  "attempts": 1,
  "finish_reason": "stop",
  "error": null,
  "raw_response": "visible",
  "jpeg_kb": 639.4
}

Elapsed: 0.4 sec

VISION TEST: SUCCESS

ALL PREFLIGHT TESTS PASSED
Starting dataset screening...

Rows to process: 4,880

Checkpoint: C:\Users\aanand37\qwen_visibility\head_visibility_fixed.jsonl
Records already in checkpoint: 0


Checking heads: 100%|███████████████████████████████████| 4880/4880 [15:41<00:00,  5.18img/s, ok=4879, fail=1, cache=0]


SCREENING SUMMARY
Images requested:       4,880
Processed this run:     4,880
Loaded from checkpoint: 0
Successful model calls: 4,879
Failed model calls:     1

Successful classification counts:
head_visibility
visible        4537
not_visible     342
Name: count, dtype: int64

Most common errors:
error
FileNotFoundError: Image not found: Q:\aanand37\Manuscript\Snapshot Wisconsin\data\Deer_clipped\SSWI000000025575326A.jpg    1
Name: count, dtype: int64

Total screening time: 15.7 minutes

FILES SAVED

Results:
C:\Users\aanand37\qwen_visibility\screening_results_fixed_20260928_134605.csv

Dataset + predictions:
C:\Users\aanand37\qwen_visibility\dataset_with_head_visibility_20260928_134605.csv

Failed requests:
C:\Users\aanand37\qwen_visibility\failed_requests_20260928_134605.csv

Checkpoint:
C:\Users\aanand37\qwen_visibility\head_visibility_fixed.jsonl

FINAL COUNTS

Status:
status
ok        4879
failed       1
Name: count, dtype: int64

Head visibility:
head_visibility
visible        4

,row_id,filename,head_visibility,status,fallback,attempts,finish_reason,error,raw_response,jpeg_kb
0,0,SSWI000000027884533B.jpg,visible,ok,False,1,stop,None,visible,639.4
1,1,SSWI000000019469717A.jpg,visible,ok,False,1,stop,None,visible,639.1
2,2,SSWI000000020749648B.jpg,not_visible,ok,False,1,stop,None,not_visible,298.8
3,3,SSWI000000010695719B.jpg,visible,ok,False,1,stop,None,visible,108.1
4,4,SSWI000000020110273C.jpg,visible,ok,False,1,stop,None,visible,158.7
5,5,SSWI000000011340915C.jpg,visible,ok,False,1,stop,None,visible,158.9
6,6,SSWI000000027884532C.jpg,visible,ok,False,1,stop,None,visible,521.5
7,7,SSWI000000013189074A.jpg,visible,ok,False,1,stop,None,visible,208.7
8,8,SSWI000000012449090C.jpg,visible,ok,False,1,stop,None,visible,127.9
9,9,SSWI000000030689976C.jpg,visible,ok,False,1,stop,None,visible,172.7
